# BenthiCat "SSS Pre-training": list, download a few sectors, inspect

Dataset: Harvard Dataverse `doi:10.7910/DVN/2VCN7Y` (about 1M unlabelled 384x384 side-scan sonar tiles, one 7z archive set per sector).

Run the cells in order. Do not download everything (about 54 GB for the sectors seen so far).
Send back the output of cell 2 and cell 4.

Edit `SECTORS` in cell 3 to change which sectors are fetched. Multi-part sectors (`.7z.001`, `.7z.002`, ...) need all parts downloaded; cell 3 does that.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, requests, subprocess
DOI = "doi:10.7910/DVN/2VCN7Y"
BASE = "https://dataverse.harvard.edu"
ARCH = "/content/drive/MyDrive/benthicat/pretrain_archives"
LOCAL = "/content/bc_pre"
os.makedirs(ARCH, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

## 2. List every file and size

In [ ]:
files, off = [], 0
while True:
    r = requests.get(f"{BASE}/api/datasets/:persistentId/versions/:latest/files",
                     params={"persistentId": DOI, "limit": 100, "offset": off}, timeout=60)
    r.raise_for_status()
    batch = r.json()["data"]
    if not batch: break
    files += batch; off += len(batch)
    if len(batch) < 100: break
rows = [(f.get("directoryLabel",""), f["label"], f["dataFile"]["id"], f["dataFile"]["filesize"]) for f in files]
for d, n, i, s in sorted(rows):
    print(f"{d:5} {n:14} id={i:<9} {s/1e6:9.1f} MB")
print(len(rows), "files,", sum(r[3] for r in rows)/1e9, "GB total")
json.dump(rows, open("/content/drive/MyDrive/benthicat/pretrain_index.json","w"))

## 3. Download chosen sectors (resumes if interrupted)

In [ ]:
SECTORS = ["N02", "N08", "N03"]
def fetch(fid, dest):
    have = os.path.getsize(dest) if os.path.exists(dest) else 0
    h = {"Range": f"bytes={have}-"} if have else {}
    with requests.get(f"{BASE}/api/access/datafile/{fid}", headers=h, stream=True, timeout=120) as r:
        if r.status_code == 416: return
        r.raise_for_status()
        with open(dest, "ab" if have else "wb") as f:
            for chunk in r.iter_content(1<<20): f.write(chunk)
for d, n, fid, size in sorted(rows):
    if d in SECTORS:
        dest = os.path.join(ARCH, n)
        if os.path.exists(dest) and os.path.getsize(dest) == size:
            print("have", n); continue
        print("downloading", n, f"{size/1e6:.0f} MB"); fetch(fid, dest)
        assert os.path.getsize(dest) == size, f"size mismatch {n}"

## 4. Extract and inspect

In [ ]:
import glob, collections
from PIL import Image
if subprocess.run("which 7z || which 7zz", shell=True, capture_output=True).returncode != 0:
    subprocess.run("apt-get install -y -q p7zip-full", shell=True)
for s in SECTORS:
    first = os.path.join(ARCH, f"{s}.7z.001")
    out = os.path.join(LOCAL, s); os.makedirs(out, exist_ok=True)
    res = subprocess.run(["7z", "x", "-y", f"-o{out}", first], capture_output=True, text=True)
    if res.returncode != 0:
        print(s, "7z FAILED:", res.stdout[-500:], res.stderr[-500:])
        continue
    allf = [p for p in glob.glob(out + "/**/*", recursive=True) if os.path.isfile(p)]
    ext = collections.Counter(os.path.splitext(p)[1].lower() for p in allf)
    print(s, len(allf), "files", dict(ext))
    imgs = [p for p in allf if p.lower().endswith((".png",".jpg",".jpeg",".tif",".tiff"))][:5]
    for p in imgs:
        im = Image.open(p); print("  ", os.path.relpath(p, out), im.size, im.mode)